# hn-time-capsule · 可复现分析 notebook

> 36 小时内一个人用 Claude Opus 4.5 写完的 LLM 应用工程教科书

**完整报告**：[`hn-time-capsule.html`](hn-time-capsule.html)（27K / ~7.2K 字）

```bash
git clone https://github.com/karpathy/hn-time-capsule ~/auto-research/hn-time-capsule
```

## 执行摘要
- 15 commits 全部在 **2025-12-09 → 12-10 的 36 小时内**
- 两次坐下：Dec 9 下午 6 小时 10 commits + Dec 10 早上 1.5 小时 5 commits
- 核心架构：**5 阶段磁盘缓存 pipeline**（fetch → prompt → analyze → parse → render）
- 最值得学的一条 commit：`76da7d5 tune prompt again`——在 9 行 diff 里浓缩了 LLM 应用工程三件套
- 发布姿态："I don't intend to support it"（CLAUDE.md 也被主动删）

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime
REPO = Path.home() / 'auto-research' / 'hn-time-capsule'
def git(*args): return subprocess.check_output(['git', '-C', str(REPO), *args], text=True)
print('commits:', git('rev-list', '--count', 'HEAD').strip())

## 1. 分钟级 vibe code 节奏

验证 "10 commits 6 小时内、中位间隔 <10 分钟" 的核心证据。

In [ ]:
log = git('log', '--reverse', '--date=format:%Y-%m-%d %H:%M', '--pretty=format:%h|%ad|%s')
rows = [l.split('|', 2) for l in log.strip().splitlines()]
prev = None
gaps = []
for h, d, s in rows:
    ts = datetime.strptime(d, '%Y-%m-%d %H:%M')
    gap = int((ts - prev).total_seconds() / 60) if prev else None
    if gap is not None: gaps.append(gap)
    tag = f'+{gap:>4}m' if gap is not None else '  start'
    print(f'  {h}  {d}  {tag}  {s}')
    prev = ts

import statistics
print(f'\n中位间隔: {statistics.median(gaps)} 分钟')
print(f'最短: {min(gaps)} 分钟   最长: {max(gaps)} 分钟')

**学到的节奏**：一次描述 → agent 写 → 验证 → 小改 → commit → 下一个。任何人想练 LLM-pair-programming 节奏，把这组时间戳当 reference。

## 2. 核心 commit: `76da7d5 tune prompt again`

LLM 应用工程三件套浓缩在 9 行 diff 里：
1. 给 LLM 加结构约束（显式 "6 sections"）
2. 加 grounding 步骤（先 summary 再评价）
3. parser 对格式漂移宽容（正则加 `\\d+[\\.\\)]` 前缀容错）

In [ ]:
print(git('show', '76da7d5', '--stat').split('\n\n', 1)[0])

In [ ]:
# 关键 diff 片段
diff = git('show', '76da7d5', '--format=', '--', 'pipeline.py')
for l in diff.splitlines():
    if l.startswith('+') or l.startswith('-'):
        print(l)

## 3. 5 阶段 pipeline 架构

验证代码里的 5 个 stage 函数都存在，对应 5 个 subcommand。

In [ ]:
# grep stage_ 函数定义
out = subprocess.check_output(
    ['grep', '-n', '^def stage_', str(REPO / 'pipeline.py')], text=True)
print(out)

每个 stage 的产物都落到 `./data/<date>/<stage>/`，只有 stage 3 (analyze) 会消耗 LLM API quota。改 prompt → 重跑 2-5；改 parser → 只重跑 4-5。**昂贵阶段永远不必重跑**。

## 4. CLAUDE.md 的生与死

对比 autoresearch（`program.md` 是核心资产，9 次修改）vs hn-time-capsule（`CLAUDE.md` 被主动删）。

In [ ]:
# CLAUDE.md 的生命轨迹
life = git('log', '--all', '--diff-filter=AD', '--pretty=format:%h %ad %s', '--date=short', '--', 'CLAUDE.md')
print(life)

In [ ]:
# 作者当时怎么想的
print(git('show', '-s', '--format=%B', 'bdde346'))

**关键对比**：

| Repo | agent-instruction 的定位 | 处置 |
|---|---|---|
| autoresearch | 产品的一部分（agent 是运行时组件） | commit + 9 次迭代 |
| hn-time-capsule | 作者的私人开发工具 | 主动删除 |

**学到的**：CLAUDE.md 该不该 commit，取决于 agent 在你项目里是"产品一部分"还是"开发工具"。两种都对。

## 5. 发布与不维护的声明

In [ ]:
# README 是否显式声明 vibe code + 不维护
readme = (REPO / 'README.md').read_text()
for line in readme.splitlines():
    if 'vibe' in line.lower() or "don't intend" in line.lower() or 'as is' in line.lower():
        print('  ', line)

## 6. Takeaways

1. **36 小时工具的发布姿态**：预设"一日工具" + 显式声明 vibe code + 不维护——这三条一起能把 issue tracker 心智负担降到 0。
2. **5 阶段磁盘缓存**是可直接抄的架构模板。任何含 LLM 调用的 pipeline 都应该照抄。
3. **一条 commit 学 LLM 应用工程**：`76da7d5` 是 9 行 diff 的 LLM engineering 101。
4. **CLAUDE.md 删除对比 program.md 保留** 揭示了同一个人在不同项目里对 agent 工作流的 4 种立场。

完整分析见 [`hn-time-capsule.html`](hn-time-capsule.html)。